# 1. Project Introduction

This notebook demonstrates a full Retrieval-Augmented Generation (RAG) pipeline for research documents. The workflow covers document loading, chunking, embeddings, vector database creation, retrieval experiments, hybrid retrieval, answer generation, evaluation, and final conclusions.

# 2. Import Libraries

These imports provide the tools for document processing, embeddings, retrieval, vector storage, and LLM answer generation.

In [2]:
import os
import sys
from pathlib import Path
from typing import List, Dict, Any

import numpy as np
import pandas as pd

from dotenv import load_dotenv
from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.documents import Document

try:
    from langchain.retrievers import EnsembleRetriever
except ImportError:
    try:
        from langchain_classic.retrievers import EnsembleRetriever
    except ImportError:
        EnsembleRetriever = None

load_dotenv()
print('Libraries imported successfully.')
print('OpenAI API key present:', bool(os.environ.get('OPENAI_API_KEY')))


Libraries imported successfully.
OpenAI API key present: True


# 3. Load Documents

Load the research documents from the docs directory and inspect a few records to confirm the source content.

In [3]:
project_root = Path.cwd()
docs_dir = project_root / 'docs'
print(f'Project root: {project_root}')
print(f'Docs directory exists: {docs_dir.exists()}')

loader = DirectoryLoader(
    str(docs_dir),
    glob='*.txt',
    loader_cls=lambda path: TextLoader(str(path), encoding='utf-8')
)
documents = loader.load()
print(f'Total documents loaded: {len(documents)}')
documents[:2]

Project root: d:\Projects\Capstone1
Docs directory exists: True
Total documents loaded: 1


[Document(metadata={'source': 'd:\\Projects\\Capstone1\\docs\\dataset-2.txt'}, page_content='Generative AI for Visualization: State of the Art and\nFuture Directions\n\narXiv:2404.18144v1 [cs.LG] 28 Apr 2024\n\nYilin Yea,b , Jianing Haoa , Yihan Houa , Zhan Wanga , Shishi Xiaoa , Yuyu\nLuoa,b , Wei Zenga,b\na\n\nThe Hong Kong University of Science and Technology\n(Guangzhou), Guangzhou, Guangdong, China\nb\nThe Hong Kong University of Science and Technology, Hong Kong SAR, China\n\nAbstract\nGenerative AI (GenAI) has witnessed remarkable progress in recent years\nand demonstrated impressive performance in various generation tasks in different domains such as computer vision and computational design. Many\nresearchers have attempted to integrate GenAI into visualization framework,\nleveraging the superior generative capacity for different operations. Concurrently, recent major breakthroughs in GenAI like diffusion model and large\nlanguage model have also drastically increase the potent

# 4. Explore Dataset

Inspect the dataset structure, document size, and high-level content patterns before building chunks and embeddings.

In [4]:
doc_lengths = [len(doc.page_content) for doc in documents]
print(f'Min length: {min(doc_lengths)}')
print(f'Max length: {max(doc_lengths)}')
print(f'Average length: {sum(doc_lengths) / len(doc_lengths):.2f}')

sample_df = pd.DataFrame({
    'source': [doc.metadata.get('source', 'unknown') for doc in documents],
    'chars': doc_lengths
})
sample_df.head()

Min length: 152639
Max length: 152639
Average length: 152639.00


,source,chars
0,d:\Projects\Capstone1\docs\dataset-2.txt,152639


# 5. Chunking

Split the source texts into smaller overlapping chunks so retrieval can focus on relevant passages rather than whole documents.

In [5]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=['\n\n', '\n', ' ', '']
)
chunks = text_splitter.split_documents(documents)
print(f'Total chunks created: {len(chunks)}')
chunks[0]

Total chunks created: 430


Document(metadata={'source': 'd:\\Projects\\Capstone1\\docs\\dataset-2.txt'}, page_content='Generative AI for Visualization: State of the Art and\nFuture Directions\n\narXiv:2404.18144v1 [cs.LG] 28 Apr 2024\n\nYilin Yea,b , Jianing Haoa , Yihan Houa , Zhan Wanga , Shishi Xiaoa , Yuyu\nLuoa,b , Wei Zenga,b\na\n\nThe Hong Kong University of Science and Technology\n(Guangzhou), Guangzhou, Guangdong, China\nb\nThe Hong Kong University of Science and Technology, Hong Kong SAR, China')

# 6. Embedding Generation

Generate embeddings for each chunk to enable semantic search and similarity comparisons.

In [6]:
embeddings_model = OpenAIEmbeddings(model='text-embedding-3-small')
sample_embedding = embeddings_model.embed_query('What is hybrid retrieval?')
print(f'Embedding dimension: {len(sample_embedding)}')
sample_embedding[:5]

Embedding dimension: 1536


[0.0306396484375,
 0.017608642578125,
 0.0311279296875,
 0.010498046875,
 0.0224456787109375]

# 7. Vector Database Creation

Store the chunk embeddings in ChromaDB to support vector search and hybrid retrieval.

In [7]:
persist_dir = project_root / 'db' / 'chroma_db'
persist_dir.mkdir(parents=True, exist_ok=True)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings_model,
    persist_directory=str(persist_dir),
    collection_metadata={'hnsw:space': 'cosine'}
)
print(f'Vector DB created at: {persist_dir}')
print(f'Collection count: {vectorstore._collection.count()}')

Vector DB created at: d:\Projects\Capstone1\db\chroma_db
Collection count: 1290


# 8. Retrieval Experiments

Compare retrieval strategies with semantic vector search and inspect the top documents for a test query.

In [8]:
query = 'How does hybrid search improve retrieval accuracy?'
retriever = vectorstore.as_retriever(search_kwargs={'k': 5})
retrieved_docs = retriever.invoke(query)
print(f'Retrieved docs: {len(retrieved_docs)}')
for i, doc in enumerate(retrieved_docs, 1):
    print(f'\n--- Result {i} ---')
    print(doc.page_content[:500])

Retrieved docs: 5

--- Result 1 ---
visualization. For example, for infographics generation, we can take the best
of both worlds by combining the previous retrieval-based methods [54, 161]
with the latest purely GenAI methods [22]. In this way, users can benefit from both the reliable real-world examples and the creativity of GenAI
models.
Multi-modal composed retrieval. WYTIWYR [57] introduces a retrieval prototype with the novel composed query which combines image input

--- Result 2 ---
visualization. For example, for infographics generation, we can take the best
of both worlds by combining the previous retrieval-based methods [54, 161]
with the latest purely GenAI methods [22]. In this way, users can benefit from both the reliable real-world examples and the creativity of GenAI
models.
Multi-modal composed retrieval. WYTIWYR [57] introduces a retrieval prototype with the novel composed query which combines image input

--- Result 3 ---
visualization. For example, for infographics g

# 9. Hybrid Retrieval

Combine vector retrieval and BM25-style keyword retrieval to improve recall and robustness.

In [9]:
# Build a BM25 retriever from the same chunk corpus
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 5

hybrid = EnsembleRetriever(
    retrievers=[vectorstore.as_retriever(search_kwargs={'k': 5}), bm25_retriever],
    weights=[0.7, 0.3]
)

hybrid_results = hybrid.invoke(query)
print(f'Hybrid result count: {len(hybrid_results)}')
for i, doc in enumerate(hybrid_results[:3], 1):
    print(f'\n--- Hybrid {i} ---')
    print(doc.page_content[:500])

Hybrid result count: 7

--- Hybrid 1 ---
visualization. For example, for infographics generation, we can take the best
of both worlds by combining the previous retrieval-based methods [54, 161]
with the latest purely GenAI methods [22]. In this way, users can benefit from both the reliable real-world examples and the creativity of GenAI
models.
Multi-modal composed retrieval. WYTIWYR [57] introduces a retrieval prototype with the novel composed query which combines image input

--- Hybrid 2 ---
this corpus to locate their desired chart using similarity search, which has
also recently incorporated some GenAI techniques.
6.1. Visualization Retrieval
Having established a set of well-crafted visualization charts and share online, the subsequent question that arises is how we can help users in searching
38

--- Hybrid 3 ---
[59] Q. Chen, S. Cao, J. Wang, N. Cao, How does automation shape the
process of narrative visualization: A survey of tools, IEEE Transactions
on Visualization and Comput

# 10. RAG Answer Generation

Use the retrieved passages to produce an answer with an LLM.

In [10]:
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0.2)
context = '\n\n'.join(doc.page_content for doc in hybrid_results[:3])
prompt = f'''
Answer the question using only the provided context.

Question: {query}

Context:
{context}
'''
response = llm.invoke(prompt)
print(response.content)

Hybrid search improves retrieval accuracy by combining reliable retrieval-based methods with the creativity of GenAI models. This approach allows users to benefit from both real-world examples and innovative solutions, enhancing the effectiveness of the search process.


# 11. Evaluation

Evaluate retrieval quality and answer usefulness by checking relevance, coverage, and consistency.

In [11]:
retrieval_quality = {
    'vector_count': len(retrieved_docs),
    'hybrid_count': len(hybrid_results),
    'query': query,
    'top_chunk_preview': hybrid_results[0].page_content[:300] if hybrid_results else ''
}

print('Retrieval quality summary:')
for key, value in retrieval_quality.items():
    print(f'{key}: {value}')

Retrieval quality summary:
vector_count: 5
hybrid_count: 7
query: How does hybrid search improve retrieval accuracy?
top_chunk_preview: visualization. For example, for infographics generation, we can take the best
of both worlds by combining the previous retrieval-based methods [54, 161]
with the latest purely GenAI methods [22]. In this way, users can benefit from both the reliable real-world examples and the creativity of GenAI
mo


# 12. Conclusion

This notebook shows the full lifecycle of a research document RAG pipeline. The vector store supports semantic retrieval, while hybrid retrieval improves robustness by combining dense and lexical signals. Final answers are grounded in the retrieved evidence, making the output more reliable and explainable.